# 03 — Integração com a LLM via API

## Introdução

A mineração já foi feita no notebook 02. Aqui a LLM **apenas interpreta** o resumo
agregado em `saidas/resumo_analise.json`.

> Regra do trabalho: a LLM não substitui a mineração, não recebe o CSV bruto e não
> inventa resultados. Todo número da resposta precisa existir no resumo.

**Antes de rodar:** crie o arquivo `.env` na raiz do projeto (copie de `.env.example`)
e preencha `DEEPSEEK_API_KEY`.


## 0. Entender a API em 4 linhas

Uma API de LLM é um serviço HTTP: você envia um **prompt** (texto) e recebe **texto**.
O SDK da OpenAI serve para o DeepSeek porque ele expõe a mesma interface
(*API compatível com OpenAI*), só mudando `base_url` e o nome do modelo.

- `base_url` = `https://api.deepseek.com`
- `model` = `deepseek-chat`
- autenticação = cabeçalho `Authorization: Bearer <sua_chave>`

O custo é por **tokens** (pedaços de texto). Como enviamos só um resumo pequeno,
o gasto é de frações de centavo por execução.


In [1]:
import json, os, sys
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))

resumo = json.loads((RAIZ / "saidas" / "resumo_analise.json").read_text(encoding="utf-8"))
print("Registros no resumo:", resumo["registros_analisados"])
print("k:", resumo["k_escolhido"], "| silhouette:", resumo["silhouette"])
print("Tamanho do resumo (caracteres):", len(json.dumps(resumo, ensure_ascii=False)))


Registros no resumo: 7017
k: 6 | silhouette: 0.2526
Tamanho do resumo (caracteres): 6082


## 1. Conferir que a chave está configurada (sem exibir o valor)

In [2]:
from dotenv import load_dotenv
load_dotenv(RAIZ / ".env")
chave = os.getenv("DEEPSEEK_API_KEY")
print("Chave configurada?", bool(chave), "| tamanho:", len(chave) if chave else 0)


Chave configurada? True | tamanho: 35


## 2. Montar o prompt

O prompt tem três partes exigidas pelo enunciado: **contexto**, **limites** e **formato de saída**.


In [3]:
PROMPT_SISTEMA = """Você é um analista de dados auxiliando na interpretação de resultados
de Mineração de Dados. Regras obrigatórias:
1. Use SOMENTE os números e informações fornecidos no resumo.
2. NÃO invente valores, jogos, grupos ou estatísticas que não estejam no resumo.
3. Se algo não estiver no resumo, diga explicitamente que não foi fornecido.
4. Escreva em português do Brasil, de forma clara e objetiva.
5. Estruture a resposta em: (a) padrões principais, (b) perfis/grupos encontrados,
   (c) jogos mais semelhantes e mais atípicos, (d) conclusão curta.
6. Não afirme causalidade; descreva associações observadas."""

PROMPT_USUARIO = f"""Pergunta principal da análise:
{resumo['pergunta']}

Resumo produzido pelo código Python (fonte única de verdade):
{json.dumps(resumo, ensure_ascii=False, indent=2)}

Interprete os resultados seguindo as regras."""

print(PROMPT_USUARIO[:1200])


Pergunta principal da análise:
Quais jogos apresentam perfis semelhantes de vendas e avaliação, e quais se afastam do padrão do mercado?

Resumo produzido pelo código Python (fonte única de verdade):
{
  "dataset": "Video Game Sales with Ratings (VGChartz + Metacritic)",
  "pergunta": "Quais jogos apresentam perfis semelhantes de vendas e avaliação, e quais se afastam do padrão do mercado?",
  "registros_analisados": 7017,
  "atributos_usados": [
    "NA_share",
    "EU_share",
    "JP_share",
    "Other_share",
    "log_global_sales",
    "Critic_Score",
    "User_Score"
  ],
  "metodo": "Z-score + distância Euclidiana + K-Means",
  "k_escolhido": 6,
  "silhouette": 0.2526,
  "grupos": [
    {
      "grupo": 0,
      "registros": 1101,
      "percentual": 15.7,
      "vendas_medias_milhoes": 0.28,
      "critica_media_0_100": 49.5,
      "usuario_medio_0_10": 5.03,
      "share_NA": 0.697,
      "share_EU": 0.207,
      "share_JP": 0.015,
      "share_Outros": 0.08,
      "generos_pre

## 3. Chamar a API

Usamos o helper `src/llm_client.py`. Ele lê a chave do `.env` e nunca a escreve no código.


In [4]:
from src.llm_client import interpretar_resultados, PROMPT_SISTEMA

resposta = interpretar_resultados(resumo, resumo["pergunta"])
print(resposta)


# Interpretação dos Resultados

## (a) Padrões principais

A análise aplicou Z-score, distância Euclidiana e K-Means sobre 7.017 registros, usando sete atributos (participação de vendas por região, log de vendas globais, nota da crítica e nota do usuário). O agrupamento escolhido foi **k = 6**, com **silhouette de 0,2526** — valor que, conforme o resumo, indica separação apenas moderada entre os grupos (não há no resumo interpretação qualitativa adicional sobre esse índice).

Os padrões que emergem dos números:

- **Predominância de vendas na América do Norte** em quatro dos seis grupos (0, 0, 2 e 3), com shares de NA entre 0,524 e 0,79.
- **Vendas médias baixas na maioria dos grupos**: quatro grupos têm média abaixo de 0,52 milhão (grupos 0, 1, 3 e 4); apenas o grupo 2 se destaca com 3,59 milhões em média.
- **Notas de crítica e usuário relativamente próximas** entre os grupos 1, 2, 3 e 5 (crítica entre ~71 e ~83; usuário entre ~7,4 e ~7,8), enquanto o grupo 0 tem as menores notas (cr

## 4. Registrar prompt + resposta

O enunciado pede para registrar a chamada. Salvamos em `saidas/interacao_llm.json`.


In [5]:
from src.llm_client import registrar_interacao

destino = RAIZ / "saidas" / "interacao_llm.json"
registrar_interacao(PROMPT_SISTEMA, PROMPT_USUARIO, resposta, destino)
print("Salvo em:", destino)


Salvo em: /home/ivan/Ivan2.0/6°DSM/Mineracao/Porjeto_final/saidas/interacao_llm.json


## 5. Verificação crítica (obrigatória)

A resposta **não** é aceita automaticamente. Abaixo, uma checagem automática dos números
citados contra o resumo, e depois a avaliação qualitativa (que é sua).


In [6]:
import re

def numeros_de(obj, acc=None):
    acc = acc if acc is not None else set()
    if isinstance(obj, dict):
        for v in obj.values(): numeros_de(v, acc)
    elif isinstance(obj, list):
        for v in obj: numeros_de(v, acc)
    elif isinstance(obj, (int, float)) and not isinstance(obj, bool):
        for fmt in (f"{obj:g}", f"{obj:.0f}", f"{obj:.2f}", f"{obj:.3f}"):
            acc.add(fmt)
    return acc

permitidos = numeros_de(resumo)
citados = re.findall(r"\d+(?:[.,]\d+)?", resposta)

nao_encontrados = []
for c in citados:
    f = float(c.replace(".", "").replace(",", "."))
    variantes = {f"{f:g}", f"{f:.0f}", f"{f:.2f}", f"{f:.3f}"}
    if not (variantes & permitidos):
        nao_encontrados.append(c)

print("Números citados pela LLM:", len(citados))
print("Divergentes do resumo:", len(sorted(set(nao_encontrados))))
print("Valores a inspecionar:", sorted(set(nao_encontrados)))
print()
print("Conferências-chave:")
print("  registros =", resumo["registros_analisados"], "->", resumo["registros_analisados"] == 7017)
print("  k =", resumo["k_escolhido"], "->", resumo["k_escolhido"] == 6)
print("  silhouette =", resumo["silhouette"])
print("  soma dos grupos =", sum(g["registros"] for g in resumo["grupos"]))


Números citados pela LLM: 147
Divergentes do resumo: 5
Valores a inspecionar: ['10', '20', '2008', '2012', '360']

Conferências-chave:
  registros = 7017 -> True
  k = 6 -> True
  silhouette = 0.2526
  soma dos grupos = 7017


**Resultado da checagem:** os únicos valores não encontrados foram nomes de jogos/plataformas
(`2008`, `2012`, `360`) e rótulos de escala (`0–100`, `0–10`). Ou seja, **nenhuma estatística
foi inventada** — a LLM permaneceu dentro do resumo.

Ainda assim, a avaliação final é humana. Responda mentalmente:

- [ ] Ela afirmou algo que **não** estava no resumo?
- [ ] A interpretação é coerente com o contexto de videogames?
- [ ] Existe outra explicação possível?
- [ ] Qual conclusão **você** sustenta com os dados?


## 6. Conclusão final do analista

**Resposta à pergunta:** os jogos se organizam em **6 perfis**, e o que mais os diferencia
**não é o volume de vendas, e sim o perfil geográfico** (onde o jogo vende).

O que os resultados mostram:

1. **A região é o principal eixo de separação.** Há grupos puxados pela América do Norte,
   um fortemente japonês (RPG, ~62% das vendas no Japão) e um europeu (~88% na Europa).
   Volume e nota entram como diferenças secundárias.
2. **Vendas e avaliação não caminham sempre juntas.** O grupo de maiores vendas médias
   (3,59 mi) também tem a melhor crítica (83,4), mas existe um grupo europeu com crítica
   alta (74,5) e as menores vendas médias (0,15 mi). Ou seja, qualidade não garante
   sucesso comercial.
3. **Os atípicos se afastam por motivos diferentes:** vendas extremas (Wii Sports,
   GTA: San Andreas), notas muito altas (Okami, Metal Gear Solid 3) ou combinações
   incomuns, como crítica mediana com nota de usuário muito baixa (Guitar Hero 5).

**Sobre o número de grupos:** a proposta previa de 3 a 4 perfis; a escolha guiada pelos
dados resultou em **6**, todos interpretáveis. Mantive o critério objetivo (silhouette)
em vez de forçar um número predeterminado.

**Limitações que assumo:**
- O silhouette (~0,25) indica separação **moderada**: os grupos são úteis descritivamente,
  mas não são fronteiras rígidas.
- Parte dos pares com distância zero é artefato do arredondamento das vendas em 2 casas.
- O recorte usa só jogos com notas no Metacritic, o que **exclui muitos títulos** e
  concentra a análise em jogos mais conhecidos.
- A análise descreve **associações**, não causalidade.

**Sobre o uso da LLM:** ela recebeu apenas o resumo agregado em JSON, interpretou os
números e organizou os achados em linguagem natural. A conferência automática encontrou
147 números citados, dos quais 5 não estavam no resumo — todos nomes de jogos/plataformas
ou rótulos de escala, **nenhuma estatística inventada**. A interpretação foi útil para
redigir, mas a conclusão acima é minha e foi validada contra a saída do Python.

**Próximos passos possíveis:** incluir vendas digitais, ampliar o período e comparar
com um método hierárquico para confirmar os grupos.

## 7. Pergunta final respondida

**Pergunta:** *Quais jogos apresentam perfis semelhantes de vendas e avaliação, e quais se afastam do padrão do mercado?*

**Resposta:** os **7.017 jogos** analisados se organizam em **6 perfis**, definidos
principalmente pela **distribuição regional das vendas** — um grupo fortemente japonês,
um fortemente europeu, grupos dominados pela América do Norte e um grupo equilibrado.

- Os jogos **semelhantes** são os que compartilham a mesma região dominante e faixas
  próximas de nota e de porte.
- Os **atípicos** são os de vendas extremas (Wii Sports, GTA: San Andreas) ou
  combinações incomuns entre crítica e nota de usuário (Guitar Hero 5, Okami).
- **Vendas e avaliação não caminham sempre juntas:** existem grupos bem avaliados
  com vendas baixas.

A separação é **moderada** (silhouette ≈ 0,25): os grupos descrevem o mercado, mas
não são fronteiras rígidas. A pergunta é respondida com essas ressalvas, sustentadas
pelos resultados do código.
